# 01 — Data Exploration

Explore the synthetic SFT, DPO, and eval gold datasets.
Goals:
- Understand tier distribution and intent frequency
- Inspect example structure before training
- Identify any data quality issues early

Run `make prepare-data` before executing this notebook to populate `data/processed/`.

In [ ]:
import json
import pathlib
from collections import Counter

import matplotlib.pyplot as plt

DATA_DIR = pathlib.Path("../data")

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

print("Data directory:", DATA_DIR.resolve())

## SFT data

In [ ]:
sft_records = load_jsonl(DATA_DIR / "sample_sft.jsonl")
print(f"SFT examples: {len(sft_records)}")
print()

# Inspect structure of the first example
example = sft_records[0]
print("Messages in example 0:")
for msg in example["messages"]:
    role = msg["role"].upper()
    preview = msg["content"][:120].replace("\n", " ")
    print(f"  [{role}] {preview}...")

In [ ]:
# Parse the assistant responses and inspect the structured outputs
def get_assistant_json(record):
    for msg in record["messages"]:
        if msg["role"] == "assistant":
            try:
                return json.loads(msg["content"])
            except json.JSONDecodeError:
                return None
    return None

sft_outputs = [get_assistant_json(r) for r in sft_records]
print("Extracted structured outputs:")
for i, out in enumerate(sft_outputs):
    if out:
        print(f"\n  Example {i}:")
        print(f"    tier:    {out.get('memory_tier')}")
        print(f"    valence: {out.get('emotional_valence')}")
        print(f"    intent:  {out.get('stated_intent')}")
        print(f"    cluster: {out.get('topic_cluster')}")

## Eval gold data — distribution analysis

In [ ]:
eval_records = load_jsonl(DATA_DIR / "eval_gold.jsonl")
print(f"Eval examples: {len(eval_records)}")

# Tier distribution
tier_counts = Counter(r["gold"]["memory_tier"] for r in eval_records)
valence_counts = Counter(r["gold"]["emotional_valence"] for r in eval_records)
tag_counts = Counter(tag for r in eval_records for tag in r.get("eval_tags", []))

print("\nTier distribution:")
for tier, count in sorted(tier_counts.items()):
    bar = "█" * count
    print(f"  {tier:<20} {bar} ({count})")

print("\nValence distribution:")
for val, count in sorted(valence_counts.items()):
    print(f"  {val:<20} {count}")

print("\nEval tag coverage:")
for tag, count in sorted(tag_counts.items()):
    print(f"  {tag:<35} {count}")

In [ ]:
# Visualize tier distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

tiers = list(tier_counts.keys())
tier_vals = list(tier_counts.values())
axes[0].barh(tiers, tier_vals, color="#4A90D9")
axes[0].set_title("Eval Gold: Tier Distribution")
axes[0].set_xlabel("Count")

valences = list(valence_counts.keys())
val_vals = list(valence_counts.values())
colors = {"positive": "#5CB85C", "neutral": "#9E9E9E", "negative": "#D9534F", "mixed": "#F0AD4E"}
bar_colors = [colors.get(v, "#4A90D9") for v in valences]
axes[1].bar(valences, val_vals, color=bar_colors)
axes[1].set_title("Eval Gold: Valence Distribution")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Aspiration vs. commitment cases

The hardest classification boundary: examples tagged `aspiration_vs_commitment`.

In [ ]:
avc_cases = [
    r for r in eval_records
    if "aspiration_vs_commitment" in r.get("eval_tags", [])
]

print(f"Aspiration vs. commitment cases: {len(avc_cases)}")
for r in avc_cases:
    print(f"\n  [{r['id']}] → {r['gold']['memory_tier'].upper()}")
    print(f"    Input:  {r['input'][:100]}...")
    if r.get("note"):
        print(f"    Note:   {r['note']}")

## DPO preference data

In [ ]:
dpo_records = load_jsonl(DATA_DIR / "sample_dpo.jsonl")
print(f"DPO preference pairs: {len(dpo_records)}")
print()

for i, pair in enumerate(dpo_records):
    print(f"── Pair {i} " + "─" * 50)
    # Show just the input portion of the prompt
    prompt_preview = pair["prompt"].split("Input:")[-1].strip()[:100]
    print(f"  Input:    {prompt_preview}...")

    chosen = json.loads(pair["chosen"])
    rejected = json.loads(pair["rejected"])

    print(f"  Chosen  → tier={chosen.get('memory_tier')}, intent={chosen.get('stated_intent') is not None}")
    print(f"  Rejected→ tier={rejected.get('memory_tier')}, intent={rejected.get('stated_intent') is not None}")
    print()